In [ ]:
# ============================================================
# AMPds2 — Full Feature Engineering Pipeline (LOCAL VERSION v2)
# Sept-Nov 2012 subset (3 months) | Synthetic anomaly injection |
# TSFresh + FRESH-pruning (real labels) | Context vector
# ============================================================

import pandas as pd
import numpy as np
import os
from tsfresh.utilities.dataframe_functions import roll_time_series, impute
from tsfresh.feature_extraction import extract_features, EfficientFCParameters, MinimalFCParameters
from tsfresh import select_features

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DATA_DIR = r'C:\1.Revanth\Projects\research'
DATE_START = '2012-09-01'
DATE_END   = '2012-10-31'          # 2 months 
WINDOW_SIZE_MIN = 15
STRIDE_MIN      = WINDOW_SIZE_MIN
APPLIANCE_COLS  = ['FRE', 'HPE', 'DWE', 'CWE', 'WOE', 'B1E']
USE_MINIMAL_FEATURES = False

N_JOBS = 10
print(f"CPUs available: {os.cpu_count()} | using n_jobs={N_JOBS}")

CACHE_DIR = os.path.join(DATA_DIR, 'pipeline_cache')   
os.makedirs(CACHE_DIR, exist_ok=True)
ROLLED_PATH       = os.path.join(CACHE_DIR, 'df_rolled.parquet')
RAW_FEATURES_PATH = os.path.join(CACHE_DIR, 'raw_features.parquet')

np.random.seed(42)

# ============================================================
# 1. LOAD DATA
# ============================================================
whe     = pd.read_csv(os.path.join(DATA_DIR, 'Electricity_WHE.csv'))
p_all   = pd.read_csv(os.path.join(DATA_DIR, 'Electricity_P.csv'))
weather = pd.read_csv(os.path.join(DATA_DIR, 'Climate_HourlyWeather.csv'))

# ============================================================
# 2. TIMESTAMP ALIGNMENT
# ============================================================
whe_ts_col = 'unix_ts' if 'unix_ts' in whe.columns else 'UNIX_TS'
p_ts_col   = 'UNIX_TS' if 'UNIX_TS' in p_all.columns else 'unix_ts'

whe['timestamp']   = pd.to_datetime(whe[whe_ts_col], unit='s', utc=True).dt.tz_convert('America/Vancouver')
p_all['timestamp'] = pd.to_datetime(p_all[p_ts_col], unit='s', utc=True).dt.tz_convert('America/Vancouver')

whe   = whe.drop(columns=[whe_ts_col]).set_index('timestamp').sort_index()
p_all = p_all.drop(columns=[p_ts_col]).set_index('timestamp').sort_index()

weather['timestamp'] = pd.to_datetime(weather['Date/Time'], format='mixed')
weather = weather.set_index('timestamp').sort_index()
weather.index = weather.index.tz_localize('America/Vancouver', ambiguous='NaT', nonexistent='shift_forward')
weather = weather[weather.index.notna()]

weather = weather.drop(columns=[
    'Data Quality', 'Temp Flag', 'Dew Point Temp Flag', 'Rel Hum Flag',
    'Wind Dir Flag', 'Wind Spd Flag', 'Visibility Flag', 'Stn Press Flag',
    'Hmdx', 'Hmdx Flag', 'Wind Chill', 'Wind Chill Flag'
], errors='ignore')

# ============================================================
# 3. FILTER TO 3-MONTH SUBSET
# ============================================================
whe     = whe.loc[DATE_START:DATE_END]
p_all   = p_all.loc[DATE_START:DATE_END]
weather = weather.loc[DATE_START:DATE_END]

print(f"WHE rows: {len(whe)}, P rows: {len(p_all)}, weather rows: {len(weather)}")

# ============================================================
# 4. MERGE WHE (mains) + selected appliances from P.csv
# ============================================================
appliance_cols_present = [c for c in APPLIANCE_COLS if c in p_all.columns]
merged = whe.join(p_all[appliance_cols_present], how='inner')

BEHAVIOR_COLS = ['V', 'I', 'P', 'Q', 'S'] + appliance_cols_present

for col in BEHAVIOR_COLS:
    merged[col] = pd.to_numeric(merged[col], errors='coerce').astype('float64')
merged[BEHAVIOR_COLS] = merged[BEHAVIOR_COLS].ffill().bfill()

# ============================================================
# 5. WEATHER: encode regime + merge into minute-level data
# ============================================================
def simplify_weather(w):
    if pd.isna(w):
        return 'Unknown'
    w = w.lower()
    if 'thunder' in w:
        return 'Thunderstorm'
    if 'snow' in w:
        return 'Snow'
    if 'fog' in w:
        return 'Fog'
    if 'rain' in w or 'drizzle' in w:
        return 'Rain'
    if 'cloud' in w:
        return 'Cloudy'
    if 'clear' in w or 'sunny' in w:
        return 'Clear'
    return 'Other'

weather['weather_regime'] = weather['Weather'].apply(simplify_weather)
print("Weather regime distribution:")
print(weather['weather_regime'].value_counts())

weather_dummies = pd.get_dummies(weather['weather_regime'], prefix='wx', dtype=int)
weather = weather.join(weather_dummies)

weather_numeric_cols = ['Temp (C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)']
for col in weather_numeric_cols:
    weather[col] = pd.to_numeric(weather[col], errors='coerce')

weather_regime_cols = weather_dummies.columns.tolist()

merged = merged.join(weather[weather_numeric_cols + weather_regime_cols], how='left')
merged[weather_numeric_cols + weather_regime_cols] = merged[weather_numeric_cols + weather_regime_cols].ffill().infer_objects(copy=False)

# ============================================================
# 6. SYNTHETIC ANOMALY INJECTION — expanded (contextual / sequential / collective / point)
# ============================================================
merged['window_id'] = merged.index.floor(f'{WINDOW_SIZE_MIN}min')
anomaly_log = []

# --- contextual ---------------------------------------------------------

def inject_heating_on_warm_day(df, log, n_events=65):
    candidates = df[(df['wx_Clear'] == 1) & (df['Temp (C)'] > 15)].index
    if len(candidates) == 0:
        return df
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'HPE'] = df.loc[window, 'HPE'].max() * np.random.uniform(3, 5) + 500
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'heating_on_warm_day'})
    return df

def inject_appliance_unusual_hours(df, log, appliance='WOE', n_events=80):
    candidates = df[(df.index.hour >= 2) & (df.index.hour <= 5)].index
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    base = df[appliance].quantile(0.9)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, appliance] = base * np.random.uniform(1.5, 3)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'appliance_unusual_hours'})
    return df

def inject_high_usage_low_occupancy(df, log, n_events=80):
    candidates = df[(df.index.hour >= 10) & (df.index.hour <= 15) & (df.index.dayofweek < 5)].index
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'P'] = df.loc[window, 'P'] + np.random.uniform(700, 1400)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'high_usage_low_occupancy'})
    return df

def inject_weekend_pattern_on_weekday(df, log, n_events=65):
    weekday_candidates = df[df.index.dayofweek < 5].index
    weekend_avg = df[df.index.dayofweek >= 5]['CWE'].mean()
    chosen = np.random.choice(weekday_candidates, size=min(n_events, len(weekday_candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'CWE'] = weekend_avg * np.random.uniform(1.5, 2.5)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'weekend_pattern_on_weekday'})
    return df

def inject_weekday_pattern_on_weekend(df, log, n_events=65):
    weekend_candidates = df[df.index.dayofweek >= 5].index
    weekday_avg = df[df.index.dayofweek < 5]['CWE'].mean()
    chosen = np.random.choice(weekend_candidates, size=min(n_events, len(weekend_candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'CWE'] = weekday_avg * np.random.uniform(1.5, 2.5)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'weekday_pattern_on_weekend'})
    return df

# --- sequential -----------------------------------------------------------

def inject_stuck_appliance_on(df, log, appliance='DWE', n_events=16, duration_min=90):
    candidates = df.index[:-duration_min]
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    stuck_value = df[appliance].quantile(0.75)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=duration_min))]
        df.loc[window, appliance] = stuck_value
        for w in pd.date_range(ts, periods=duration_min // WINDOW_SIZE_MIN, freq=f'{WINDOW_SIZE_MIN}min'):
            log.append({'window_id': w.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'stuck_appliance_on'})
    return df

def inject_stuck_appliance_off(df, log, appliance='HPE', n_events=16, duration_min=90):
    candidates = df.index[:-duration_min]
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=duration_min))]
        df.loc[window, appliance] = 0.0
        for w in pd.date_range(ts, periods=duration_min // WINDOW_SIZE_MIN, freq=f'{WINDOW_SIZE_MIN}min'):
            log.append({'window_id': w.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'stuck_appliance_off'})
    return df

def inject_gradual_drift(df, log, appliance='B1E', n_events=20, duration_min=60):
    candidates = df.index[:-duration_min]
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=duration_min))]
        direction = np.random.choice([1, -1])
        ramp = np.linspace(1.0, 1.0 + direction * np.random.uniform(0.8, 1.5), len(window))
        df.loc[window, appliance] = df.loc[window, appliance] * ramp
        label = 'gradual_drift_increase' if direction == 1 else 'gradual_drift_decrease'
        for w in pd.date_range(ts, periods=duration_min // WINDOW_SIZE_MIN, freq=f'{WINDOW_SIZE_MIN}min'):
            log.append({'window_id': w.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': label})
    return df

def inject_sustained_overload(df, log, n_events=16, duration_min=60):
    candidates = df.index[:-duration_min]
    chosen = np.random.choice(candidates, size=min(n_events, len(candidates)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=duration_min))]
        df.loc[window, 'P'] = df.loc[window, 'P'] + np.random.uniform(1000, 1800)
        for w in pd.date_range(ts, periods=duration_min // WINDOW_SIZE_MIN, freq=f'{WINDOW_SIZE_MIN}min'):
            log.append({'window_id': w.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'sustained_overload'})
    return df

# --- collective -------------------------------------------------------

def inject_multiple_high_power_simultaneous(df, log, appliances=('HPE', 'DWE', 'CWE'), n_events=40):
    chosen = np.random.choice(df.index, size=min(n_events, len(df.index)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        for app in appliances:
            df.loc[window, app] = df[app].quantile(0.9) * np.random.uniform(1.2, 1.8)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'multiple_high_power_simultaneous'})
    return df

def inject_impossible_appliance_combo(df, log, appliances=('HPE', 'WOE'), n_events=33):
    chosen = np.random.choice(df.index, size=min(n_events, len(df.index)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        for app in appliances:
            df.loc[window, app] = df[app].quantile(0.95) * np.random.uniform(1.3, 2.0)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'impossible_appliance_combo'})
    return df

# --- point (kept minority) ---------------------------------------------

def inject_power_spike(df, log, n_events=40):
    chosen = np.random.choice(df.index, size=min(n_events, len(df.index)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'P'] = df.loc[window, 'P'] + np.random.uniform(900, 1700)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'power_spike'})
    return df

def inject_sensor_glitch(df, log, n_events=26):
    chosen = np.random.choice(df.index, size=min(n_events, len(df.index)), replace=False)
    for ts in chosen:
        window = df.index[(df.index >= ts) & (df.index < ts + pd.Timedelta(minutes=WINDOW_SIZE_MIN))]
        df.loc[window, 'V'] = df.loc[window, 'V'] * np.random.uniform(0.5, 1.8)
        df.loc[window, 'I'] = df.loc[window, 'I'] * np.random.uniform(0.3, 2.5)
        log.append({'window_id': ts.floor(f'{WINDOW_SIZE_MIN}min'), 'anomaly_type': 'sensor_glitch'})
    return df

# run all injectors — contextual first (highest priority), point last (minority)
merged = inject_heating_on_warm_day(merged, anomaly_log)
merged = inject_appliance_unusual_hours(merged, anomaly_log)
merged = inject_high_usage_low_occupancy(merged, anomaly_log)
merged = inject_weekend_pattern_on_weekday(merged, anomaly_log)
merged = inject_weekday_pattern_on_weekend(merged, anomaly_log)
merged = inject_stuck_appliance_on(merged, anomaly_log)
merged = inject_stuck_appliance_off(merged, anomaly_log)
merged = inject_gradual_drift(merged, anomaly_log)
merged = inject_sustained_overload(merged, anomaly_log)
merged = inject_multiple_high_power_simultaneous(merged, anomaly_log)
merged = inject_impossible_appliance_combo(merged, anomaly_log)
merged = inject_power_spike(merged, anomaly_log)
merged = inject_sensor_glitch(merged, anomaly_log)

merged = merged.drop(columns=['window_id'])

anomaly_df = pd.DataFrame(anomaly_log).drop_duplicates(subset='window_id')
print("\nInjected anomaly counts by type:")
print(anomaly_df['anomaly_type'].value_counts())
print(f"Total anomalous windows: {len(anomaly_df)}")

anomaly_df.to_csv(os.path.join(CACHE_DIR, 'injected_anomaly_labels.csv'), index=False)

# ============================================================
# 7. BUILD RAW MULTIVARIATE STREAM -> LONG FORMAT FOR TSFRESH
# ============================================================
long_df = merged.reset_index().melt(
    id_vars=['timestamp'],
    value_vars=BEHAVIOR_COLS,
    var_name='sensor', value_name='value'
)
long_df['entity_id'] = 'house'

# ============================================================
# 8. SLIDING WINDOW — cached locally
# ============================================================
if os.path.exists(ROLLED_PATH):
    print("Found cached df_rolled — loading instead of recomputing")
    df_rolled = pd.read_parquet(ROLLED_PATH)
else:
    print("No cache found — running roll_time_series (slow step #1)")
    df_rolled = roll_time_series(
        long_df,
        column_id='entity_id',
        column_sort='timestamp',
        column_kind='sensor',
        max_timeshift=WINDOW_SIZE_MIN - 1,
        min_timeshift=WINDOW_SIZE_MIN - 1,
        rolling_direction=STRIDE_MIN
    )
    df_rolled['id'] = df_rolled['id'].astype(str)
    df_rolled.to_parquet(ROLLED_PATH)
    print(f"Saved to {ROLLED_PATH}")

print(f"Total windows to extract: {df_rolled['id'].nunique()}")

# ============================================================
# 9. TSFRESH FEATURE EXTRACTION — cached locally
# ============================================================
if os.path.exists(RAW_FEATURES_PATH):
    print("Found cached raw_features — loading instead of re-extracting")
    raw_features = pd.read_parquet(RAW_FEATURES_PATH)
else:
    print("No cache found — running extract_features (slow step #2, this is the big one)")
    fc_params = MinimalFCParameters() if USE_MINIMAL_FEATURES else EfficientFCParameters()
    raw_features = extract_features(
        df_rolled,
        column_id='id', column_sort='timestamp',
        column_kind='sensor', column_value='value',
        default_fc_parameters=fc_params,
        n_jobs=N_JOBS,
        chunksize=500
    )
    raw_features = impute(raw_features)

    def parse_window_id(idx_str):
        ts_str = idx_str.split("Timestamp('")[1].split("'")[0]
        return pd.Timestamp(ts_str).floor(f'{WINDOW_SIZE_MIN}min')

    raw_features.index = pd.Index([parse_window_id(i) for i in raw_features.index], name='window_id')
    raw_features = raw_features[~raw_features.index.duplicated(keep='first')]

    raw_features.to_parquet(RAW_FEATURES_PATH)
    print(f"Saved to {RAW_FEATURES_PATH}")

print(raw_features.shape)
print(raw_features.index[:5])

# ============================================================
# 10. BUILD REAL LABEL (from injected anomalies)
# ============================================================
true_target = pd.Series(0, index=raw_features.index)
true_target.loc[true_target.index.isin(anomaly_df['window_id'])] = 1
true_target = true_target.astype(int)

print("Label distribution:")
print(true_target.value_counts())

label_lookup = anomaly_df.set_index('window_id')['anomaly_type']

# ============================================================
# 11. FRESH PRUNING -> BEHAVIOR VECTOR
# ============================================================
behavior_vector = select_features(raw_features, true_target)
print(f"\nFRESH-pruned behavior features: {behavior_vector.shape[1]} (from {raw_features.shape[1]})")

# ============================================================
# 12. CONTEXT VECTOR (cyclic time + weather regime)
# ============================================================
agg_dict = {col: 'mean' for col in weather_numeric_cols}
agg_dict.update({col: 'max' for col in weather_regime_cols})

context = merged.resample(f'{WINDOW_SIZE_MIN}min').agg(agg_dict)

context['hour']      = context.index.hour
context['dayofweek'] = context.index.dayofweek

context['hour_sin']   = np.sin(2 * np.pi * context['hour'] / 24)
context['hour_cos']   = np.cos(2 * np.pi * context['hour'] / 24)
context['dow_sin']    = np.sin(2 * np.pi * context['dayofweek'] / 7)
context['dow_cos']    = np.cos(2 * np.pi * context['dayofweek'] / 7)
context['is_weekend'] = context['dayofweek'].isin([5, 6]).astype(int)

context = context.drop(columns=['hour', 'dayofweek'])

# ============================================================
# 13. ALIGN + CONCATENATE BEHAVIOR + CONTEXT + LABELS
# ============================================================
combined = behavior_vector.join(context, how='inner')
combined = combined.dropna()

combined['is_anomaly'] = combined.index.isin(anomaly_df['window_id']).astype(int)
combined['anomaly_type'] = combined.index.map(label_lookup).fillna('normal')

print(f"\nFinal combined feature matrix: {combined.shape[0]} windows x {combined.shape[1]} columns")
print(combined['is_anomaly'].value_counts())
print(f"Anomaly rate: {combined['is_anomaly'].mean()*100:.2f}%")

# ============================================================
# 14. SAVE FINAL OUTPUT
# ============================================================
output_path = os.path.join(CACHE_DIR, 'ampds_behavior_context_labeled_features.csv')
combined.to_csv(output_path)
print(f"Saved: {output_path}")

CPUs available: 22 | using n_jobs=10
WHE rows: 87840, P rows: 87840, weather rows: 1464
Weather regime distribution:
weather_regime
Clear     737
Cloudy    469
Rain      163
Fog        95
Name: count, dtype: int64


C:\Users\revan\AppData\Local\Temp\ipykernel_4756\4290664632.py:121: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  merged[weather_numeric_cols + weather_regime_cols] = merged[weather_numeric_cols + weather_regime_cols].ffill().infer_objects(copy=False)



Injected anomaly counts by type:
anomaly_type
stuck_appliance_on                  89
stuck_appliance_off                 88
appliance_unusual_hours             78
high_usage_low_occupancy            77
heating_on_warm_day                 63
weekend_pattern_on_weekday          63
weekday_pattern_on_weekend          61
sustained_overload                  53
gradual_drift_increase              40
multiple_high_power_simultaneous    34
power_spike                         34
impossible_appliance_combo          31
sensor_glitch                       24
gradual_drift_decrease              23
Name: count, dtype: int64
Total anomalous windows: 758
No cache found — running roll_time_series (slow step #1)


Rolling: 100%|██████████| 55/55 [01:23<00:00,  1.53s/it]


Saved to C:\1.Revanth\Projects\research\pipeline_cache\df_rolled.parquet
Total windows to extract: 5856
No cache found — running extract_features (slow step #2, this is the big one)


Feature Extraction: 100%|██████████| 129/129 [07:20<00:00,  3.41s/it]
c:\1.Revanth\Projects\research\.venv\Lib\site-packages\tsfresh\utilities\dataframe_functions.py:198: RuntimeWarning: The columns <ArrowStringArray>
[                               'I__partial_autocorrelation__lag_7',
                                'I__partial_autocorrelation__lag_8',
                                'I__partial_autocorrelation__lag_9',
                                   'I__spkt_welch_density__coeff_8',
                                 'I__ar_coefficient__coeff_0__k_10',
                                 'I__ar_coefficient__coeff_1__k_10',
                                 'I__ar_coefficient__coeff_2__k_10',
                                 'I__ar_coefficient__coeff_3__k_10',
                                 'I__ar_coefficient__coeff_4__k_10',
                                 'I__ar_coefficient__coeff_5__k_10',
 ...
   'HPE__agg_linear_trend__attr_"slope"__chunk_len_50__f_agg_"var"',
  'HPE__agg_linear

Saved to C:\1.Revanth\Projects\research\pipeline_cache\raw_features.parquet
(5856, 8547)
DatetimeIndex(['2012-09-01 00:00:00-07:00', '2012-09-01 00:15:00-07:00',
               '2012-09-01 00:30:00-07:00', '2012-09-01 00:45:00-07:00',
               '2012-09-01 01:00:00-07:00'],
              dtype='datetime64[us, UTC-07:00]', name='window_id', freq=None)
Label distribution:
0    5098
1     758
Name: count, dtype: int64

FRESH-pruned behavior features: 816 (from 8547)

Final combined feature matrix: 5856 windows x 832 columns
is_anomaly
0    5098
1     758
Name: count, dtype: int64
Anomaly rate: 12.94%
Saved: C:\1.Revanth\Projects\research\pipeline_cache\ampds_behavior_context_labeled_features.csv
